---

# ~ NVIDIA Nemotron Model Reasoning Challenge ~
*Enhancing LLM reasoning capabilities using LoRA and Chain-of-Thought (CoT)*

---

## Table of Contents

1. [**Initialization**](#1)
2. [**Data Curation & SFT Formatting**](#2)
3. [**Model Initialization & LoRA Configuration**](#3)
4. [**Training Execution & Model Archiving**](#4)
5. [**Output Format Validation**](#5)
6. [**Packaging for Kaggle Submission**](#6)

---

# Introduction <a name="1"></a>

---

The **NVIDIA Nemotron Model Reasoning Challenge** is a generative AI fine-tuning task: enhance the advanced reasoning capabilities of the Nemotron-3-Nano-30B model using a curated Chain-of-Thought (CoT) dataset and Low-Rank Adaptation (LoRA). Performance is evaluated based on the model's accuracy in solving complex logic problems, with a strict metric requirement that the final answer must be formatted inside a `\boxed{}` syntax. The submission notebook must execute completely offline without internet access, requiring efficient GPU resource management for both training and inference.

---

## Approach

The **Nemotron-3-Nano-30B** base model serves as the foundation. We employ Parameter-Efficient Fine-Tuning (PEFT) using **LoRA (Low-Rank Adaptation)** on a specialized Chain-of-Thought (CoT) dataset to enhance logical reasoning. The entire pipeline is optimized for constrained hardware, focusing on memory-efficient training and deterministic, formatted generation.

```
Nemotron-3-Nano-30B (Frozen Base Weights)
    |
    +-- LoRA Adapter (rank=32, target_modules="all-linear")
    |        |
    |        +-- Supervised Fine-Tuning (SFT) with CoT Dataset
    |        |        |
    |        |        v
    |        |   Merged Model (Base + LoRA)
    |        |
    +-- Instructional Prompting (Enforcing \boxed{} syntax)
             |
             v
         Deterministic Generation (Temperature=0.0)
             |
             v
         Response String Parsing -> evaluation
```

---

# Initialization <a name="2"></a>

---

The runtime environment is configured by enforcing GPU availability, defining global directory paths, and performing silent offline installations of critical dependencies like `triton`, `trl`, and `mamba_ssm`. System-level monkey patches and CUDA runtime linkers are dynamically applied to ensure Nemotron architecture compatibility within the internet-disabled Kaggle environment.

## Environment

In [ ]:
import os, platform, sys
print('Python :', sys.version)
print('OS     :', platform.system(), platform.release())
print('CWD    :', os.getcwd())

---
## Install 

In [ ]:
import subprocess
import sys
import glob
from pathlib import Path

# Mendefinisikan pola pencarian untuk wheel yang diperlukan
WHEEL_PATTERNS = [
    "triton*.whl",
    "datasets-*.whl",
    "trl-*.whl",
    "causal*conv1d*.whl",
    "mamba_ssm-*.whl"
]

# Mencari lokasi wheel di input Kaggle
found_wheels = []
for pattern in WHEEL_PATTERNS:
    matches = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if matches:
        found_wheels.append(matches[-1])

if len(found_wheels) >= len(WHEEL_PATTERNS):
    # Instalasi Offline jika semua wheel ditemukan
    for wheel in found_wheels:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheel], check=True)
    print(f'Successfully installed {len(found_wheels)} packages from Kaggle dataset wheels.')
else:
    # Fallback ke instalasi Online jika wheel tidak lengkap
    packages = ["datasets", "trl", "causal-conv1d", "mamba-ssm"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
    print('Installed packages from PyPI (Internet Access Required).')

---
## Libraries & Patching

In [ ]:
import os
import zipfile
import os
import json
import pandas as pd
import re
import random
from datasets import Dataset as HFDataset
import pandas as pd
import re
import random
import os
import sys
import glob
import types
import ctypes
import torch
import torch.nn.functional as F
import datasets
import trl
import mamba_ssm
import triton
import kagglehub
import json
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

# Injecting mock modules to handle Mamba3 import issues in Nemotron architecture
# This is a critical fix for the Nemotron-3-Nano model on Kaggle
for _mod_name in [
    'mamba_ssm.modules.mamba3',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
]:
    sys.modules[_mod_name] = types.ModuleType(_mod_name)

try:
    sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None
except KeyError:
    pass

# Forcefully link CUDA runtime to prevent compilation crashes during training
try:
    cuda_paths = glob.glob("/usr/local/cuda*/lib64/libcudart.so.12")
    if cuda_paths:
        ctypes.CDLL(cuda_paths[0], mode=ctypes.RTLD_GLOBAL)
        print(f"Successfully linked CUDA runtime: {cuda_paths[0]}")
except Exception as e:
    print(f"Note: CUDA linking skipped or failed: {e}")

---
## Settings

In [ ]:
class Settings:
    # 1. Environment & Hardware Validation
    IS_GPU = torch.cuda.is_available()
    DEVICE = "cuda" if IS_GPU else "cpu"
    if not IS_GPU:
        print("Warning: GPU not detected. Performance will be limited.")

    # 2. Path Configurations
    COMPETITION_DIR = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge"
    COT_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection/train_split_with_cot.csv"
    OUTPUT_DIR = "/kaggle/working/sft_adapter"
    
    # 3. Model & Tokenizer Settings
    MODEL_ID = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    MAX_SEQ_LEN = 2048
    
    # 4. LoRA Hyperparameters
    LORA_RANK = 32
    LORA_ALPHA = 16
    LORA_DROPOUT = 0.05
    
    # 5. Training Hyperparameters
    BATCH_SIZE = 1
    GRAD_ACCUM = 4
    LEARNING_RATE = 2e-4
    NUM_EPOCHS = 1
    SEED = 42

# Initialize Output Directory
os.makedirs(Settings.OUTPUT_DIR, exist_ok=True)

# Set Seed for Reproducibility
torch.manual_seed(Settings.SEED)
if Settings.IS_GPU:
    torch.cuda.manual_seed_all(Settings.SEED)

print(f"Settings initialized. Device: {Settings.DEVICE}")

# Data Curation & SFT Formatting

---
## Tokenizer Initialization & Path Resolution

This cell uses the KaggleHub library to dynamically resolve and download the local cached path of the Nemotron base model, preventing hardcoded path errors if the Kaggle environment shifts file locations. It then initializes the appropriate tokenizer directly from this resolved path and explicitly assigns the end-of-sequence token as the padding token, which is a crucial requirement to prevent tensor alignment issues during the batch generation process.

In [ ]:
RESOLVED_MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

tokenizer = AutoTokenizer.from_pretrained(RESOLVED_MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

___
## Dataset Ingestion & Stratified Sampling

This section handles the ingestion and balancing of the raw training dataset. It loads the pre-generated Chain-of-Thought dataset and systematically extracts precise quantities of data based on predefined problem categories, such as bit manipulation or text encryption, to ensure the model does not overfit to a heavily skewed category. Finally, it concatenates these stratified subsets into a unified dataframe and thoroughly shuffles the rows to maintain a randomized distribution during the fine-tuning iterations.

In [ ]:
df_cot = pd.read_csv(Settings.COT_DATASET_PATH)

# --- SAKELAR METODE SAMPLING UNTUK JURNAL ---
SAMPLING_METHOD = "stratified" # UBAH STRING INI MENJADI "random" SAAT EKSPERIMEN BASELINE

if SAMPLING_METHOD == "stratified":
    print("Menerapkan metode Stratified Sampling...")
    TYPE_SAMPLES = {
        "Numeral Conversion": 300,
        "Gravitational Constant": 400,
        "Unit Conversion": 700,
        "Text Encryption": 700,
        "Bit Manipulation": 607,
        "Equation Transformation": 200,
    }

    sampled_dfs = []
    for ptype, n_samples in TYPE_SAMPLES.items():
        subset = df_cot[df_cot["type"] == ptype]
        sampled = subset if n_samples >= len(subset) else subset.sample(n=n_samples, random_state=Settings.SEED)
        sampled_dfs.append(sampled)

    train_df = pd.concat(sampled_dfs, ignore_index=True).sample(frac=1, random_state=Settings.SEED).reset_index(drop=True)
    
    # HAPUS keyword 'global', langsung definisikan variabelnya
    output_csv_name = "metrics_stratified.csv"

elif SAMPLING_METHOD == "random":
    print("Menerapkan metode Random Sampling murni...")
    # Total dari metode stratified: 300+400+700+700+607+200 = 2907 sampel
    TOTAL_SAMPLES = 2907 
    
    # Mengambil sampel secara acak dari seluruh populasi dataset
    train_df = df_cot.sample(n=TOTAL_SAMPLES, random_state=Settings.SEED).reset_index(drop=True)
    
    # HAPUS keyword 'global', langsung definisikan variabelnya
    output_csv_name = "metrics_random.csv"
    
records = []
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

for _, row in train_df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot = str(row["generated_cot"])

    if not cot or cot == "nan" or len(cot.strip()) < 5:
        continue

    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()

    user_content = prompt + PROMPT_SUFFIX
    assistant_content = f"{cot_cleaned}\n</think>\n\\boxed{{{answer}}}"

    records.append({
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    })

dataset = HFDataset.from_list(records)
print(f"SFT Dataset created with {len(records)} records.")

___
## SFT Message Formatting & Cleaning

This cell iterates through the curated dataset to construct a standardized conversational structure compatible with the instruction-tuning framework. It injects a strict instructional suffix into every user prompt demanding that the final answer be enclosed in specific bounding box syntax, which is a mandatory metric evaluation criterion for the competition. It simultaneously cleans the reasoning steps by removing any duplicate box wrappers, wraps the final extracted answer accurately, and compiles the entire structured payload into a Hugging Face Dataset object primed for the trainer.

In [ ]:
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
records = []

for _, row in train_df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot = str(row["generated_cot"])
    
    if not cot or cot == "nan" or len(cot.strip()) < 5:
        continue
        
    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
    user_content = prompt + PROMPT_SUFFIX
    assistant_content = f"{cot_cleaned}\n</think>\n\\boxed{{{answer}}}"
    
    records.append({
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    })

hf_train_dataset = HFDataset.from_list(records)

# Model Initialization & LoRA Configuration

___
## Base Model Loading & Memory Optimization

This cell is responsible for loading the heavy base architecture of the Nemotron language model. It dynamically maps the model layers across the available GPU memory and forces the computation to use the `bfloat16` data type, which significantly reduces VRAM consumption while preserving numerical stability during gradient updates. Additionally, it iterates through the system modules to explicitly disable the model's native fast path execution, a critical preventative measure to avoid internal compilation bugs associated with the Triton backend when running on Kaggle infrastructure.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    RESOLVED_MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Successfully patched fast-path for: {name}")

___
## LoRA (Low-Rank Adaptation) Architecture

This segment configures the Low-Rank Adaptation (LoRA) parameters to enable parameter-efficient fine-tuning. By targeting all linear modules (`all-linear`) with a relatively high rank of 32, the configuration ensures that the model can capture and learn complex reasoning patterns required for the competition without the computationally prohibitive cost of updating the entire 30 billion parameters of the base model. The configuration is then wrapped around the base model, freezing the original weights and activating only the specified LoRA matrices for training.

In [ ]:
target_modules = ["in_proj", "out_proj", "up_proj", "down_proj"]

lora_config = LoraConfig(
    r=Settings.LORA_RANK,
    lora_alpha=Settings.LORA_ALPHA,
    target_modules=target_modules,
    lora_dropout=Settings.LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

This final initialization cell defines the rigid hyperparameters required for the Supervised Fine-Tuning (SFT) process. It sets the maximum sequence length to 2048 tokens to comfortably accommodate the lengthy, multi-step Chain-of-Thought (CoT) responses without truncation. To aggressively manage GPU memory, it utilizes a micro-batch size of 1 coupled with 4 gradient accumulation steps, and heavily relies on gradient checkpointing with the `use_reentrant` flag set to true to prevent out-of-memory errors during the backward pass. Finally, it instantiates the `SFTTrainer` object, binding the prepped model, curated dataset, tokenizer, and these tailored arguments together, leaving the pipeline fully primed for execution.

___
## SFT (Supervised Fine-Tuning) Orchestration

In [ ]:
import pandas as pd
import torch
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig

class StatisticalMetricsCallback(TrainerCallback):
    def __init__(self, output_filename="training_metrics.csv"):
        self.metrics_data = []
        self.output_filename = output_filename

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            vram_allocated = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0
            vram_reserved = torch.cuda.max_memory_reserved() / (1024**3) if torch.cuda.is_available() else 0
            
            log_entry = {
                "step": state.global_step,
                "loss": logs.get("loss"),
                "grad_norm": logs.get("grad_norm"),
                "learning_rate": logs.get("learning_rate"),
                "epoch": logs.get("epoch"),
                "vram_allocated_gb": round(vram_allocated, 3),
                "vram_reserved_gb": round(vram_reserved, 3)
            }
            self.metrics_data.append(log_entry)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

    def on_train_end(self, args, state, control, **kwargs):
        df = pd.DataFrame(self.metrics_data)
        df.to_csv(self.output_filename, index=False)
        print(f"Data komputasi statistik dan memori berhasil disimpan ke: {self.output_filename}")

training_args = SFTConfig(
    output_dir=Settings.OUTPUT_DIR,
    per_device_train_batch_size=Settings.BATCH_SIZE,
    gradient_accumulation_steps=Settings.GRAD_ACCUM,
    num_train_epochs=Settings.NUM_EPOCHS,
    learning_rate=Settings.LEARNING_RATE,
    logging_steps=1,  # <--- SUDAH DIREVISI MENJADI 1
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    max_length=Settings.MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[StatisticalMetricsCallback(output_filename=output_csv_name)] # <--- SUDAH DITAMBAHKAN
)

# Training Execution & Model Archiving

___
## Memory Lifecycle Management

This cell orchestrates the core fine-tuning loop. It begins with proactive memory management by clearing the CUDA cache and enabling `expandable_segments` to prevent fragmentation on the 30B model's large activation tensors. The training is wrapped in a monitoring block to track execution time and safely catch hardware-related failures during the gradient descent process.

In [ ]:
torch.cuda.empty_cache()
import gc
gc.collect()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import time
start_time = time.time()

try:
    trainer.train()
    training_time = time.time() - start_time
    print(f"Training completed in {training_time / 60:.2f} minutes.")
except Exception as e:
    raise RuntimeError(f"Training failed: {e}")

___
## Model Archiving & Config Patching

After training, this cell persists the learned LoRA adapters and tokenizer to the output directory. It performs critical post-processing on the `adapter_config.json` by hardcoding the base model path and enabling inference mode. These patches are mandatory for the Kaggle offline evaluator to correctly load the model weights without attempting to reach external repositories.

In [ ]:
trainer.model.save_pretrained(Settings.OUTPUT_DIR)
tokenizer.save_pretrained(Settings.OUTPUT_DIR)

config_path = os.path.join(Settings.OUTPUT_DIR, "adapter_config.json")

with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

print(f"Model and patched config saved successfully to: {Settings.OUTPUT_DIR}")

# Output Format Validation

---
## Resource Reclamation

This cell explicitly deletes the bulky training objects from the global namespace and forces a deep garbage collection alongside a CUDA cache clearing. This crucial cleanup prevents immediate out-of-memory errors when transitioning from training to inference. It then reloads a fresh instance of the base Nemotron model into the GPU and merges it seamlessly with the newly saved LoRA adapter weights using the `PeftModel` interface. Finally, it sets the merged model into evaluation mode and loads the localized tokenizer from the output directory to ensure absolute parity with the training phase.

In [ ]:
if 'trainer' in globals():
    del trainer
if 'model' in globals():
    del model
import gc
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    RESOLVED_MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

model_with_lora = PeftModel.from_pretrained(base_model, Settings.OUTPUT_DIR)
model_with_lora.eval() 

tokenizer = AutoTokenizer.from_pretrained(Settings.OUTPUT_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

---
## Model Reloading & Peft Integration

This cell performs a controlled, local inference test to validate the effectiveness of the fine-tuning process. It constructs a sample reasoning prompt appended with the exact instructional suffix used during training, formats it using the tokenizer's chat template, and pushes it to the model. The generation is strictly configured with a temperature of 0.0 and sampling disabled to guarantee deterministic outputs, simulating the rigid environment of the competition evaluator. After decoding the generated tokens, a simple conditional check verifies whether the model successfully adopted the required bounding box formatting syntax for its final answer.

In [ ]:
test_prompt = "Convert the roman numeral 'XIV' to a standard number."
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

messages = [
    {"role": "user", "content": test_prompt + PROMPT_SUFFIX}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(model_with_lora.device)

with torch.no_grad():
    outputs = model_with_lora.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = outputs[0][inputs.input_ids.shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"Prompt:\n{test_prompt}\n")
print(f"Response:\n{response}\n")

if "\\boxed{" in response:
    print("Validation Passed: The model successfully used the \\boxed{} format.")
else:
    print("Validation Failed: The \\boxed{} format was not found in the response.")

# Packaging for Kaggle Submission

---
## Integrity Check & Compression

This final packaging step is designed to format the output for direct submission to the Kaggle evaluation system. It first performs a strict validation check to ensure that both the `adapter_config.json` and the heavy `adapter_model.safetensors` files exist in the output directory, raising an explicit error if the prior training step failed to save them. It then systematically compresses these required files into a flat ZIP archive (`submission.zip`), intentionally stripping any parent folder structures using the `arcname` parameter, which is a strict formatting requirement for the competition's automated grading backend. Finally, it calculates and prints the total size of the generated submission payload for verification.

In [ ]:
required_files = ["adapter_config.json", "adapter_model.safetensors"]
missing_files = [f for f in required_files if not os.path.exists(os.path.join(Settings.OUTPUT_DIR, f))]

if missing_files:
    raise FileNotFoundError(f"Missing required adapter files: {missing_files}. Please ensure training completed successfully.")

SUBMISSION_ZIP_PATH = "/kaggle/working/submission.zip"

with zipfile.ZipFile(SUBMISSION_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        file_path = os.path.join(Settings.OUTPUT_DIR, fname)
        zf.write(file_path, arcname=fname)

zip_size_mb = os.path.getsize(SUBMISSION_ZIP_PATH) / (1024 * 1024)
print(f"Submission file created successfully: {SUBMISSION_ZIP_PATH}")
print(f"Archive Size: {zip_size_mb:.2f} MB")

# Visualization

In [ ]:
# =====================================================================
# 1. DYNAMIC DATA DETECTION & LOADING
# =====================================================================
file_stratified = "metrics_stratified.csv"
file_random = "metrics_random.csv"

has_stratified = os.path.exists(file_stratified)
has_random = os.path.exists(file_random)

if not has_stratified and not has_random:
    raise SystemExit(" No CSV data found. Please complete the training at least once.")

print("Experimental Data Status:")
print(f"  - Stratified CoT : {' AVAILABLE' if has_stratified else ' UNAVAILABLE'}")
print(f"  - Simple Random  : {' AVAILABLE' if has_random else ' UNAVAILABLE (Comparison graphs incomplete)'}\n")

df_stratified = pd.read_csv(file_stratified) if has_stratified else None
df_random = pd.read_csv(file_random) if has_random else None

# =====================================================================
# 2. JOURNAL AESTHETICS CONFIGURATION
# =====================================================================
plt.rcParams.update({
    "font.family": "serif",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 11,
    "figure.dpi": 300 
})

def smooth_curve(points, window=20):
    return points.rolling(window=window, min_periods=1).mean()

# =====================================================================
# 3. FIGURE 1: Loss Convergence & Gradient Volatility
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Subplot (a): Training Loss ---
if has_random:
    axes[0].plot(df_random['step'], df_random['loss'], alpha=0.3, color='#e74c3c', linewidth=1)
    axes[0].plot(df_random['step'], smooth_curve(df_random['loss']), label='Simple Random', color='#c0392b', linewidth=2)
if has_stratified:
    axes[0].plot(df_stratified['step'], df_stratified['loss'], alpha=0.3, color='#2980b9', linewidth=1)
    axes[0].plot(df_stratified['step'], smooth_curve(df_stratified['loss']), label='Stratified CoT', color='#2c3e50', linewidth=2)

axes[0].set_title("(a) Cross-Entropy Loss Convergence")
axes[0].set_xlabel("Training Steps")
axes[0].set_ylabel("Training Loss")
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# --- Subplot (b): Gradient Norm ---
if has_random:
    axes[1].plot(df_random['step'], df_random['grad_norm'], alpha=0.3, color='#e74c3c', linewidth=1)
    axes[1].plot(df_random['step'], smooth_curve(df_random['grad_norm']), label='Simple Random', color='#c0392b', linewidth=2)
if has_stratified:
    axes[1].plot(df_stratified['step'], df_stratified['grad_norm'], alpha=0.3, color='#2980b9', linewidth=1)
    axes[1].plot(df_stratified['step'], smooth_curve(df_stratified['grad_norm']), label='Stratified CoT', color='#2c3e50', linewidth=2)

axes[1].set_title("(b) L2 Gradient Norm Volatility")
axes[1].set_xlabel("Training Steps")
axes[1].set_ylabel("Gradient Norm")
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.savefig("loss_gradient_comparison.pdf", format='pdf', bbox_inches='tight')
plt.show()

# =====================================================================
# 4. FIGURE 2: Variance Distribution & VRAM
# =====================================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

# --- Subplot (a): KDE Plot Gradient Variance ---
if has_random:
    sns.kdeplot(data=df_random, x="grad_norm", fill=True, color='#e74c3c', label="Simple Random", ax=axes2[0])
if has_stratified:
    sns.kdeplot(data=df_stratified, x="grad_norm", fill=True, color='#2980b9', label="Stratified CoT", ax=axes2[0])
    
axes2[0].set_title("Density Distribution of Gradient Norms")
axes2[0].set_xlabel("Gradient Norm Magnitude")
axes2[0].set_ylabel("Density")
axes2[0].legend()

# --- Subplot (b): Peak VRAM Allocated ---
if has_random:
    axes2[1].plot(df_random['step'], df_random['vram_allocated_gb'], label='Simple Random', color='#e74c3c', linewidth=2)
if has_stratified:
    axes2[1].plot(df_stratified['step'], df_stratified['vram_allocated_gb'], label='Stratified CoT', color='#2980b9', linewidth=2)
    
axes2[1].set_title("VRAM Allocation Trajectory")
axes2[1].set_xlabel("Training Steps")
axes2[1].set_ylabel("VRAM Allocated (GB)")
axes2[1].grid(True, linestyle='--', alpha=0.6)
axes2[1].legend()

plt.tight_layout()
plt.savefig("variance_vram_analysis.pdf", format='pdf', bbox_inches='tight')
plt.show()

# =====================================================================
# 5. METRICS EXTRACTION FOR TABLE 2 (LATEX DRAFT)
# =====================================================================
def print_table_metrics(df, name):
    initial_step = 10 if len(df) > 10 else 0
    initial_loss = df['loss'].iloc[initial_step]
    terminal_loss = df['loss'].iloc[-1]
    mean_grad = df['grad_norm'].mean()
    var_grad = df['grad_norm'].var()
    peak_vram = df['vram_allocated_gb'].max()
    
    print(f"--- Metrics for {name} ---")
    print(f"Initial Loss (t={initial_step}) : {initial_loss:.3f}")
    print(f"Terminal Loss (final) : {terminal_loss:.3f}")
    print(f"Mean Gradient Norm    : {mean_grad:.3f}")
    print(f"Gradient Variance     : {var_grad:.3f}")
    print(f"Peak VRAM (GB)        : {peak_vram:.2f}\n")

print("\n" + "="*50)
print("CALCULATED RESULTS FOR LATEX TABLE 2")
print("="*50)
if has_random:
    print_table_metrics(df_random, "Simple Random")
if has_stratified:
    print_table_metrics(df_stratified, "Stratified CoT")